In [18]:
import pandas as pd

In [19]:
df = pd.read_csv('../backend/logs.csv')

In [20]:
df.head()

,id,colaborador_id,tag,evento,observacao,data_hora
0,1,1.0,TAG001,ENTRADA,Entrada autorizada,2026-05-12 21:47:44.874377
1,2,1.0,TAG001,SAIDA,Saida registrada,2026-05-12 21:47:44.874377
2,3,2.0,TAG002,ENTRADA,Entrada autorizada,2026-05-12 21:47:44.874377
3,4,3.0,TAG003,NEGADO,Colaborador sem permissao,2026-05-12 21:47:44.874377
4,6,NaN,TAG005,INVASAO,Tentativa de invasao com tag desconhecida,2026-05-12 21:47:44.874377


In [21]:
df['data_hora'] = pd.to_datetime(df['data_hora'])

In [22]:
df['data'] = df['data_hora'].dt.date

In [23]:
entradas = df[df['evento'] == 'ENTRADA']

entradas_por_dia = entradas.groupby('data').size()

entradas_por_dia

data
2026-05-12    2
2026-05-13    7
dtype: int64

In [24]:
saidas = df[df['evento'] == 'SAIDA']

saidas_por_dia = saidas.groupby('data').size()

saidas_por_dia

data
2026-05-12    1
2026-05-13    5
dtype: int64

In [25]:
invasoes = df[df['evento'] == 'INVASAO']

invasoes_por_dia = invasoes.groupby('data').size()

invasoes_por_dia

data
2026-05-12     2
2026-05-13    10
dtype: int64

In [26]:
negados = df[df['evento'] == 'NEGADO']

negados_por_dia = negados.groupby('data').size()

negados_por_dia

data
2026-05-12    2
2026-05-13    1
dtype: int64

In [27]:
logs_entrada_saida = df[
    df['evento'].isin(['ENTRADA', 'SAIDA'])
].copy()

In [28]:
logs_entrada_saida = logs_entrada_saida.sort_values(
    by=['colaborador_id', 'data_hora']
)

In [29]:
resultado = []

for colaborador_id in logs_entrada_saida['colaborador_id'].unique():

    logs_colaborador = logs_entrada_saida[
        logs_entrada_saida['colaborador_id'] == colaborador_id
    ]

    entrada = None
    tempo_total = pd.Timedelta(0)

    for _, row in logs_colaborador.iterrows():

        if row['evento'] == 'ENTRADA':
            entrada = row['data_hora']

        elif row['evento'] == 'SAIDA' and entrada is not None:

            tempo_total += row['data_hora'] - entrada
            entrada = None

    resultado.append({
        'colaborador_id': colaborador_id,
        'tempo_total': tempo_total
    })

In [30]:
permanencia_df = pd.DataFrame(resultado)

permanencia_df

,colaborador_id,tempo_total
0,1.0,0 days 00:00:07.488501
1,2.0,0 days 21:02:31.082741
2,6.0,0 days 00:01:01.235228
3,7.0,0 days 00:00:00


In [31]:
permanencia_df['horas'] = (
    permanencia_df['tempo_total']
    .dt.total_seconds() / 3600
)

permanencia_df

,colaborador_id,tempo_total,horas
0,1.0,0 days 00:00:07.488501,0.002080
1,2.0,0 days 21:02:31.082741,21.041967
2,6.0,0 days 00:01:01.235228,0.017010
3,7.0,0 days 00:00:00,0.000000


In [32]:
ranking_invasoes = (
    invasoes
    .groupby('tag')
    .size()
    .sort_values(ascending=False)
)

ranking_invasoes

tag
837196207282    3
15292364039     2
288338938175    2
763370409838    2
552048006075    1
TAG005          1
TAG006          1
dtype: int64

In [33]:
permanencia_df.to_csv(
    'relatorio_permanencia.csv',
    index=False
)